In [2]:
# !pip install pandas
# !pip install mysql-connector-python

In [3]:
import pandas as pd
import mysql.connector
import os

# ==========================
# MySQL Connection
# ==========================
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="datacleaning"
)

cursor = conn.cursor()

# ==========================
# CSV File Details
# ==========================
csv_file = r"H:\Data Cleaning Projects\club_member_info\csv\club_member_info.csv"
table_name = "club_member_info"

# ==========================
# Read CSV
# ==========================
print("Reading CSV...")

df = pd.read_csv(csv_file)

# Replace NaN with None
df = df.where(pd.notnull(df), None)

# ==========================
# Clean Column Names
# ==========================
df.columns = [
    col.strip()
       .replace(" ", "_")
       .replace("-", "_")
       .replace(".", "_")
       .replace("/", "_")
       .replace("(", "")
       .replace(")", "")
    for col in df.columns
]

print(f"Rows Found: {len(df)}")
print(f"Columns Found: {len(df.columns)}")

# ==========================
# Detect SQL Data Types
# ==========================
def get_sql_type(dtype):

    if pd.api.types.is_integer_dtype(dtype):
        return "INT"

    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"

    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"

    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "DATETIME"

    else:
        return "TEXT"


# ==========================
# Drop Existing Table (Optional)
# ==========================
cursor.execute(f"DROP TABLE IF EXISTS `{table_name}`")

# ==========================
# Create Table
# ==========================
columns_sql = []

for col in df.columns:
    sql_type = get_sql_type(df[col].dtype)
    columns_sql.append(f"`{col}` {sql_type}")

create_table_query = f"""
CREATE TABLE `{table_name}`
(
    {', '.join(columns_sql)}
)
"""

print("Creating Table...")

cursor.execute(create_table_query)

# ==========================
# Insert Data
# ==========================
print("Preparing Data Insert...")

column_names = ", ".join([f"`{col}`" for col in df.columns])

placeholders = ", ".join(["%s"] * len(df.columns))

insert_query = f"""
INSERT INTO `{table_name}`
({column_names})
VALUES
({placeholders})
"""

data = [
    tuple(None if pd.isna(value) else value for value in row)
    for row in df.to_numpy()
]

print("Inserting Records...")

cursor.executemany(insert_query, data)

conn.commit()

print(f"Successfully inserted {cursor.rowcount} rows into '{table_name}'")

# ==========================
# Close Connection
# ==========================
cursor.close()
conn.close()

print("Database connection closed.")

Reading CSV...
Rows Found: 2010
Columns Found: 8
Creating Table...
Preparing Data Insert...
Inserting Records...
Successfully inserted 2010 rows into 'club_member_info'
Database connection closed.
